# Hard Data Situations

Notebook for **Chapter 12** of the *ml* study document. Run the cells top to bottom. Read the full explanation on the site: https://dadipavan.github.io/Learning/study/ml/ch12.html

Tip: change a number, predict the output, then run the cell again.

## 12.1 Imbalanced Classes

In [ ]:
import numpy as np
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.dummy import DummyClassifier
from sklearn.metrics import *

X, y = make_classification(n_samples=20000, n_features=10, n_informative=5, n_redundant=0,
        weights=[0.99], flip_y=0.005, class_sep=0.8, random_state=0)
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, random_state=0, stratify=y)
print("positives train/test", ytr.sum(), yte.sum(), "rate", y.mean().round(4))
d = DummyClassifier(strategy='most_frequent').fit(Xtr, ytr)
print("always-negative acc", round(accuracy_score(yte, d.predict(Xte)), 4),
      "recall", recall_score(yte, d.predict(Xte)))

positives train/test 182 78 rate 0.013
always-negative acc 0.987 recall 0.0


In [ ]:
def rep(name, m, thr=0.5, p=None):
    p = m.predict_proba(Xte)[:, 1] if p is None else p; pr = (p >= thr).astype(int)
    print(f"{name:22s} acc {accuracy_score(yte,pr):.4f} prec {precision_score(yte,pr,zero_division=0):.3f} "
          f"rec {recall_score(yte,pr):.3f} ROC {roc_auc_score(yte,p):.3f} PR {average_precision_score(yte,p):.3f} meanp {p.mean():.4f}")
mk = lambda **k: make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000, **k))
lr = mk().fit(Xtr, ytr); rep("plain thr0.5", lr)

plain thr0.5           acc 0.9908 prec 0.960 rec 0.308 ROC 0.830 PR 0.534 meanp 0.0134


In [ ]:
from sklearn.model_selection import cross_val_predict, StratifiedKFold
pc = cross_val_predict(mk(), Xtr, ytr, cv=StratifiedKFold(5, shuffle=True, random_state=0),
                       method='predict_proba')[:, 1]
ps, rs, ts = precision_recall_curve(ytr, pc); f1 = 2*ps*rs/(ps+rs+1e-12)
t = ts[np.argmax(f1[:-1])]; print("best F1 thr from CV", round(t, 4))
rep("threshold moved", lr, thr=t)

best F1 thr from CV 0.2112
threshold moved        acc 0.9912 prec 0.791 rec 0.436 ROC 0.830 PR 0.534 meanp 0.0134


In [ ]:
lw = mk(class_weight='balanced').fit(Xtr, ytr); rep("class_weight balanced", lw)
rng = np.random.default_rng(0); pos = np.where(ytr == 1)[0]; neg = np.where(ytr == 0)[0]
idx = np.concatenate([neg, rng.choice(pos, len(neg), replace=True)])      # oversample positives
lo = mk().fit(Xtr[idx], ytr[idx]); rep("random oversample", lo)
idx = np.concatenate([pos, rng.choice(neg, len(pos)*10, replace=False)])  # undersample negatives 10:1
lu = mk().fit(Xtr[idx], ytr[idx]); rep("undersample 10:1", lu)

class_weight balanced  acc 0.8227 prec 0.055 rec 0.782 ROC 0.857 PR 0.470 meanp 0.3139
random oversample      acc 0.8237 prec 0.055 rec 0.782 ROC 0.855 PR 0.474 meanp 0.3145
undersample 10:1       acc 0.9897 prec 0.633 rec 0.487 ROC 0.847 PR 0.513 meanp 0.0629


In [ ]:
w = (len(pos)*10) / len(neg); pu = lu.predict_proba(Xte)[:, 1]
pcor = w*pu / (w*pu + (1 - pu)); rep("undersample corrected", lu, p=pcor)
pw = lw.predict_proba(Xte)[:, 1]; w2 = len(pos)/len(neg); pw2 = w2*pw/(w2*pw + (1 - pw))
for nm, pp in [('plain', lr.predict_proba(Xte)[:, 1]), ('class_weight', pw), ('class_weight corrected', pw2)]:
    print(nm, 'brier', round(brier_score_loss(yte, pp), 4))

undersample corrected  acc 0.9895 prec 0.941 rec 0.205 ROC 0.847 PR 0.513 meanp 0.0120
plain brier 0.0082
class_weight brier 0.1334
class_weight corrected brier 0.0107


In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score
Xo = np.vstack([Xtr, Xtr[ytr == 1].repeat(9, axis=0)])
yo = np.concatenate([ytr, np.ones((ytr == 1).sum()*9, int)])
c5 = StratifiedKFold(5, shuffle=True, random_state=0)
bad  = cross_val_score(RandomForestClassifier(100, random_state=0, n_jobs=-1), Xo, yo, cv=c5, scoring='average_precision')
good = cross_val_score(RandomForestClassifier(100, random_state=0, n_jobs=-1), Xtr, ytr, cv=c5, scoring='average_precision')
print("oversample BEFORE cv PR-AUC", bad.mean().round(3), "honest cv PR-AUC", good.mean().round(3))

oversample BEFORE cv PR-AUC 1.0 honest cv PR-AUC 0.512


In [ ]:
ps, rs, ts = precision_recall_curve(yte, lr.predict_proba(Xte)[:, 1])
print("recall at precision>=0.30:", rs[ps >= 0.30].max().round(3))

recall at precision>=0.30: 0.564


## 12.2 Small Data, Noisy Labels, and More Features Than Rows

In [ ]:
from sklearn.model_selection import RepeatedStratifiedKFold
rng = np.random.default_rng(1)
n = 60; Xs = rng.normal(size=(n, 5)); ys = (Xs[:, 0] + 0.8*Xs[:, 1] + rng.normal(0, 1, n) > 0).astype(int)
Xs_t = rng.normal(size=(5000, 5)); ys_t = (Xs_t[:, 0] + 0.8*Xs_t[:, 1] + rng.normal(0, 1, 5000) > 0).astype(int)
for nm, m in [('logreg', mk()), ('forest', RandomForestClassifier(200, random_state=0))]:
    m.fit(Xs, ys)
    cvv = cross_val_score(m, Xs, ys, cv=RepeatedStratifiedKFold(n_splits=5, n_repeats=20, random_state=0), scoring='roc_auc')
    print(nm, "repeated-cv", cvv.mean().round(3), "+-", cvv.std().round(3),
          "big-test", round(roc_auc_score(ys_t, m.predict_proba(Xs_t)[:, 1]), 3),
          "train", round(roc_auc_score(ys, m.predict_proba(Xs)[:, 1]), 3))
single = [cross_val_score(mk(), Xs, ys, cv=StratifiedKFold(5, shuffle=True, random_state=s), scoring='roc_auc').mean() for s in range(10)]
print("single 5-fold across 10 seeds", np.round(single, 3), "range", round(max(single) - min(single), 3))

small n=60 logreg repeated-cv 0.813 +- 0.104 big-test 0.824 train 0.869
small n=60 forest repeated-cv 0.714 +- 0.146 big-test 0.775 train 1.0
single 5-fold across 10 seeds [0.808 0.797 0.77  0.803 0.821 0.832 0.813 0.745 0.765 0.804] range 0.087


In [ ]:
from sklearn.tree import DecisionTreeClassifier
Xn, yn = make_classification(n_samples=3000, n_features=10, n_informative=5, random_state=0, class_sep=1.0, flip_y=0)
Xa, Xb, ya, yb = train_test_split(Xn, yn, test_size=0.4, random_state=0)
for noise in [0, 0.2, 0.4]:
    r = np.random.default_rng(0); yf = np.where(r.random(len(ya)) < noise, 1 - ya, ya); out = []
    for nm, m in [('logreg', mk()), ('deep tree', DecisionTreeClassifier(random_state=0)),
                  ('forest leaf10', RandomForestClassifier(200, min_samples_leaf=10, random_state=0, n_jobs=-1))]:
        m.fit(Xa, yf); out.append(f"{nm} {accuracy_score(yb, m.predict(Xb)):.3f}")
    print("noise", noise, out)

noise 0 ['logreg 0.876', 'deep tree 0.884', 'forest leaf10 0.903']
noise 0.2 ['logreg 0.872', 'deep tree 0.718', 'forest leaf10 0.913']
noise 0.4 ['logreg 0.858', 'deep tree 0.591', 'forest leaf10 0.802']


In [ ]:
from sklearn.model_selection import cross_val_predict
r = np.random.default_rng(0); yf = np.where(r.random(len(ya)) < 0.2, 1 - ya, ya); flipped = (yf != ya)
oof = cross_val_predict(mk(), Xa, yf, cv=5, method='predict_proba')[:, 1]
top = np.argsort(-np.abs(oof - yf))[:100]
print("flagged top100: truly flipped", flipped[top].sum(), "base rate", flipped.mean().round(3))

flagged top100: truly flipped 94 base rate 0.202


In [ ]:
rng = np.random.default_rng(2); n, p_ = 60, 500
Xh = rng.normal(size=(n, p_)); yh = (Xh[:, 0] + Xh[:, 1] - Xh[:, 2] + rng.normal(0, 0.5, n) > 0).astype(int)
Xh_t = rng.normal(size=(5000, p_)); yh_t = (Xh_t[:, 0] + Xh_t[:, 1] - Xh_t[:, 2] + rng.normal(0, 0.5, 5000) > 0).astype(int)
for nm, m in [('weak reg C=1e6', LogisticRegression(C=1e6, max_iter=5000)),
              ('L2 C=0.05', LogisticRegression(C=0.05, max_iter=5000)),
              ('L1 C=0.3', LogisticRegression(penalty='l1', solver='liblinear', C=0.3))]:
    m.fit(Xh, yh); print(nm, "train acc", round(m.score(Xh, yh), 3), "test acc", round(m.score(Xh_t, yh_t), 3),
                         "nonzero", int((m.coef_ != 0).sum()))

weak reg C=1e6 train acc 1.0 test acc 0.602 nonzero 500
L2 C=0.05 train acc 1.0 test acc 0.603 nonzero 500
L1 C=0.3 train acc 1.0 test acc 0.862 nonzero 19


## 12.3 Missing Data and Messy Features

In [ ]:
import pandas as pd
rng = np.random.default_rng(0); n = 5000
age = rng.integers(20, 70, n); income = np.exp(rng.normal(10.5, 0.4, n)) + 800*(age - 20)
spend = 0.02*income + 30*np.sin(age/10) + rng.normal(0, 200, n)
df = pd.DataFrame(dict(age=age, income=income, spend=spend))
def make_missing(kind):
    d = df.copy(); r = np.random.default_rng(1)
    if kind == 'MCAR': m = r.random(n) < 0.3
    elif kind == 'MAR': m = r.random(n) < np.where(d.age > 50, 0.55, 0.05)
    else: m = r.random(n) < 1/(1 + np.exp(-(d.income - 78000)/6000))
    d.loc[m, 'income'] = np.nan; return d, m
for kind in ['MCAR', 'MAR', 'MNAR']:
    d, m = make_missing(kind)
    print(kind, "missing", m.mean().round(3), "| true mean", df.income.mean().round(0),
          "| mean of observed", d.income.mean().round(0), "| true mean of missing rows", df.income[m].mean().round(0))

MCAR missing 0.304 | true mean income of all 59208.0 | mean of observed 59001.0 | true mean of missing rows 59681.0
MAR missing 0.241 | true mean income of all 59208.0 | mean of observed 56046.0 | true mean of missing rows 69143.0
MNAR missing 0.194 | true mean income of all 59208.0 | mean of observed 52592.0 | true mean of missing rows 86692.0


In [ ]:
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.compose import ColumnTransformer
from sklearn.metrics import mean_squared_error
d, m = make_missing('MAR'); tr, te = train_test_split(np.arange(n), test_size=0.3, random_state=0)
Xd = d[['age', 'income']]
def fitscore(pipe):
    pipe.fit(Xd.iloc[tr], d.spend.iloc[tr]); return mean_squared_error(d.spend.iloc[te], pipe.predict(Xd.iloc[te]))**0.5
print("baseline sd of spend", d.spend.iloc[te].std().round(1))
print("oracle (no missing) RMSE", round(mean_squared_error(df.spend.iloc[te], LinearRegression().fit(df[['age','income']].iloc[tr], df.spend.iloc[tr]).predict(df[['age','income']].iloc[te]))**0.5, 1))
print("mean impute", round(fitscore(make_pipeline(SimpleImputer(strategy='mean'), LinearRegression())), 1))
print("mean impute + indicator", round(fitscore(make_pipeline(ColumnTransformer([('i', SimpleImputer(strategy='mean', add_indicator=True), ['age', 'income'])]), LinearRegression())), 1))
print("KNN impute", round(fitscore(make_pipeline(StandardScaler(), KNNImputer(n_neighbors=10), LinearRegression())), 1))
print("iterative impute", round(fitscore(make_pipeline(IterativeImputer(random_state=0, max_iter=10), LinearRegression())), 1))
print("HistGB native NaN", round(fitscore(HistGradientBoostingRegressor(random_state=0, max_iter=100, learning_rate=0.05)), 1))

baseline sd of spend 448.1
oracle (no missing) RMSE 198.8
mean impute 278.9
mean impute + indicator 267.9
KNN impute 255.9
iterative impute 254.2
HistGB native NaN 260.5


In [ ]:
from sklearn.preprocessing import OneHotEncoder
pre = ColumnTransformer([
    ('num', make_pipeline(SimpleImputer(strategy='median'), StandardScaler()), ['age', 'income']),
    ('cat', make_pipeline(SimpleImputer(strategy='most_frequent'), OneHotEncoder(handle_unknown='ignore')), ['plan']),
])  # 'plan' is an illustrative categorical column; this block only defines the pipeline and is not run here

## 12.4 High-Cardinality Categorical Features

In [ ]:
from sklearn.preprocessing import TargetEncoder
rng = np.random.default_rng(0); n = 6000; K = 800
code = rng.integers(0, K, n); eff = rng.normal(0, 1.0, K)
dist = rng.normal(5, 2, n); z = -0.5 + eff[code] + 0.15*(dist - 5)
y = (rng.random(n) < 1/(1 + np.exp(-z))).astype(int)
D = pd.DataFrame(dict(code=code.astype(str), dist=dist))
print("unique codes", D.code.nunique(), "rows per code mean", round(n/K, 1), "rare codes (<=3 rows)", int((D.code.value_counts() <= 3).sum()))
Dtr, Dte, ytr, yte = train_test_split(D, y, test_size=0.3, random_state=0)
def ev(pre, name, model=None):
    m = make_pipeline(pre, model or LogisticRegression(max_iter=2000)).fit(Dtr, ytr)
    print(f"{name:34s} test AUC {roc_auc_score(yte, m.predict_proba(Dte)[:, 1]):.3f}  train AUC {roc_auc_score(ytr, m.predict_proba(Dtr)[:, 1]):.3f}")
ev(ColumnTransformer([('n', StandardScaler(), ['dist'])]), "distance only")
ev(ColumnTransformer([('c', OneHotEncoder(handle_unknown='ignore'), ['code']), ('n', StandardScaler(), ['dist'])]), "one-hot (800 cols), C=1")
ev(ColumnTransformer([('c', OneHotEncoder(handle_unknown='ignore'), ['code']), ('n', StandardScaler(), ['dist'])]), "one-hot, C=0.3", LogisticRegression(C=0.3, max_iter=2000))

unique codes 800 rows per code mean 7.5 rare codes (<=3 rows) 52
distance only                      test AUC 0.577  train AUC 0.568
one-hot (800 cols), C=1            test AUC 0.684  train AUC 0.841
one-hot, C=0.3                     test AUC 0.680  train AUC 0.815


In [ ]:
te_map = pd.Series(ytr, index=Dtr.index).groupby(Dtr.code).mean(); gm = ytr.mean()
Dtr_n = Dtr.copy(); Dtr_n['te'] = Dtr.code.map(te_map)
Dte_n = Dte.copy(); Dte_n['te'] = Dte.code.map(te_map).fillna(gm)
m = LogisticRegression(max_iter=2000).fit(Dtr_n[['te', 'dist']], ytr)
print("naive target encoding   test", round(roc_auc_score(yte, m.predict_proba(Dte_n[['te', 'dist']])[:, 1]), 3),
      " train", round(roc_auc_score(ytr, m.predict_proba(Dtr_n[['te', 'dist']])[:, 1]), 3))
ev(ColumnTransformer([('c', TargetEncoder(random_state=0), ['code']), ('n', 'passthrough', ['dist'])]), "sklearn TargetEncoder (cross-fit)")

naive target encoding              test AUC 0.671  train AUC 0.847
sklearn TargetEncoder (cross-fit)  test AUC 0.679  train AUC 0.838


In [ ]:
fm = Dtr.code.value_counts()
Dtr_f = Dtr.assign(freq=Dtr.code.map(fm)); Dte_f = Dte.assign(freq=Dte.code.map(fm).fillna(1))
m = LogisticRegression(max_iter=2000).fit(Dtr_f[['freq', 'dist']], ytr)
print("frequency encoding", round(roc_auc_score(yte, m.predict_proba(Dte_f[['freq', 'dist']])[:, 1]), 3))
from sklearn.feature_extraction import FeatureHasher
import scipy.sparse as sp
h = FeatureHasher(n_features=64, input_type='string')
Htr = h.transform([[c] for c in Dtr.code]); Hte = h.transform([[c] for c in Dte.code])
m = LogisticRegression(max_iter=2000).fit(sp.hstack([Htr, Dtr[['dist']].values]), ytr)
print("hashing 64 buckets", round(roc_auc_score(yte, m.predict_proba(sp.hstack([Hte, Dte[['dist']].values]))[:, 1]), 3))
print("test rows with unseen code", int((~Dte.code.isin(Dtr.code)).sum()), "of", len(Dte))

frequency encoding test AUC 0.579
hashing 64 buckets test AUC 0.570
test rows with unseen code 14 of 1800


## 12.5 Distribution Shift and Non-Stationarity

In [ ]:
from sklearn.ensemble import GradientBoostingRegressor
rng = np.random.default_rng(0)
def gen(n, temp_mu, slope):
    t = rng.normal(temp_mu, 4, n); promo = rng.integers(0, 2, n)
    return pd.DataFrame(dict(temp=t, promo=promo, orders=50 + slope*(t - 15) + 20*promo + rng.normal(0, 8, n)))
A = gen(3000, 12, 2.0); B = gen(3000, 22, 2.0); C = gen(3000, 22, 0.5)
for nm, mdl in [('linear', LinearRegression()), ('boosting', GradientBoostingRegressor(random_state=0))]:
    mdl.fit(A[['temp', 'promo']], A.orders)
    r = lambda Dd: mean_squared_error(Dd.orders, mdl.predict(Dd[['temp', 'promo']]))**0.5
    print(nm, "RMSE same period", round(r(gen(1000, 12, 2.0)), 2), "| covariate shift", round(r(B), 2), "| concept drift", round(r(C), 2))

linear RMSE same period 8.19 | covariate shift 7.99 | concept drift 14.8
boosting RMSE same period 8.12 | covariate shift 8.76 | concept drift 14.24


In [ ]:
from scipy import stats
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score, StratifiedKFold
def psi(a, b, bins=10):
    e = np.quantile(a, np.linspace(0, 1, bins + 1)); e[0] = -np.inf; e[-1] = np.inf
    pa = np.clip(np.histogram(a, e)[0]/len(a), 1e-4, None); pb = np.clip(np.histogram(b, e)[0]/len(b), 1e-4, None)
    return np.sum((pb - pa)*np.log(pb/pa))
print("PSI temp A vs fresh A", round(psi(A.temp, gen(3000, 12, 2).temp), 3), "A vs B", round(psi(A.temp, B.temp), 3))
print("KS temp A vs B", stats.ks_2samp(A.temp, B.temp).statistic.round(3))
def adv(P, Q):
    Z = pd.concat([P[['temp', 'promo']], Q[['temp', 'promo']]]); lab = np.r_[np.zeros(len(P)), np.ones(len(Q))]
    return cross_val_score(RandomForestClassifier(100, random_state=0, min_samples_leaf=5), Z, lab,
                           cv=StratifiedKFold(5, shuffle=True, random_state=0), scoring='roc_auc').mean()
print("adversarial AUC A vs fresh A", round(adv(A, gen(3000, 12, 2)), 3), "A vs B", round(adv(A, B), 3))

PSI temp A vs fresh A 0.008 A vs B 4.754
KS temp A vs B 0.797
adversarial AUC A vs fresh A 0.495 A vs B 0.952


In [ ]:
Ctest = gen(1000, 22, 0.5); mixed = pd.concat([A, C.iloc[:600]]); wts = np.r_[np.ones(3000), np.full(600, 5.0)]
for nm, (Dd, w) in {'old only': (A, None), 'old+600 new': (mixed, None),
                    'old+600 new weighted x5': (mixed, wts), 'new 600 only': (C.iloc[:600], None)}.items():
    mm = LinearRegression().fit(Dd[['temp', 'promo']], Dd.orders, sample_weight=w)
    print(nm, "RMSE on new regime", round(mean_squared_error(Ctest.orders, mm.predict(Ctest[['temp', 'promo']]))**0.5, 2))

old only RMSE on new regime 15.1
old+600 new RMSE on new regime 9.52
old+600 new weighted x5 RMSE on new regime 8.26
new 600 only RMSE on new regime 7.91


## 12.6 Multiple Imputation (MICE) and Oversampling (SMOTE), Tested Honestly

Two popular fixes appear in every tutorial: fill the missing values with a model (MICE), and fabricate extra rare-class rows (SMOTE). Both can help and both are often used wrongly. This section runs each against a clear baseline.

In [ ]:
import numpy as np
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer, SimpleImputer
rng = np.random.default_rng(0); n = 1000
age = rng.normal(40, 10, n); inc = 2 * age + rng.normal(0, 5, n); X = np.c_[age, inc]
mask = rng.random(n) < 0.3                      # income missing at random
Xm = X.copy(); Xm[mask, 1] = np.nan
true = X[mask, 1]
for name, imp in [("mean", SimpleImputer()), ("iterative", IterativeImputer(random_state=0))]:
    Z = imp.fit_transform(Xm)
    print(name, "RMSE", round(np.sqrt(np.mean((Z[mask, 1] - true) ** 2)), 2),
          "std of income", round(Z[:, 1].std(), 2), "corr(age,income)", round(np.corrcoef(Z.T)[0, 1], 3))
print("truth: std", round(X[:, 1].std(), 2), "corr", round(np.corrcoef(X.T)[0, 1], 3))
# MICE with sample_posterior: m=5 imputations, Rubin pooling of mean income
ests, vars_ = [], []
for s in range(5):
    Z = IterativeImputer(sample_posterior=True, random_state=s).fit_transform(Xm)[:, 1]
    ests.append(Z.mean()); vars_.append(Z.var(ddof=1) / n)
ests = np.array(ests); W = np.mean(vars_); B = ests.var(ddof=1); T = W + (1 + 1/5) * B
print("pooled mean", ests.mean().round(2), "within", round(W, 4), "between", round(B, 4), "total SE", round(T ** .5, 3))

mean RMSE 19.36 std of income 17.64 corr(age,income) 0.834
iterative RMSE 5.2 std of income 20.37 corr(age,income) 0.978
truth: std 20.48 corr 0.968
pooled mean 79.11 within 0.4238 between 0.0035 total SE 0.654


In [ ]:
import numpy as np
from sklearn.datasets import make_classification
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import average_precision_score, recall_score, precision_score
X, y = make_classification(4000, 8, n_informative=4, weights=[0.97], flip_y=0.01, class_sep=1.0, random_state=1)
Xtr, Xte, ytr, yte = train_test_split(X, y, stratify=y, random_state=0)
def smote(Xp, n_new, k=5, seed=0):
    r = np.random.default_rng(seed); nn = NearestNeighbors(n_neighbors=k + 1).fit(Xp)
    idx = nn.kneighbors(Xp, return_distance=False)[:, 1:]
    i = r.integers(0, len(Xp), n_new); j = idx[i, r.integers(0, k, n_new)]
    return Xp[i] + r.random((n_new, 1)) * (Xp[j] - Xp[i])
def fit_eval(Xa, ya, tag):
    m = RandomForestClassifier(200, random_state=0, n_jobs=-1).fit(Xa, ya); p = m.predict_proba(Xte)[:, 1]
    print(f"{tag:22s} PR-AUC {average_precision_score(yte, p):.3f}  @0.5: recall {recall_score(yte, p>.5):.2f} prec {precision_score(yte, p>.5):.2f}")
print("train positives", ytr.sum(), "of", len(ytr))
fit_eval(Xtr, ytr, "baseline")
Xp = Xtr[ytr == 1]; new = smote(Xp, (ytr == 0).sum() - len(Xp))
fit_eval(np.r_[Xtr, new], np.r_[ytr, np.ones(len(new), int)], "SMOTE to 50/50")
m = RandomForestClassifier(200, random_state=0, class_weight="balanced_subsample", n_jobs=-1).fit(Xtr, ytr); p = m.predict_proba(Xte)[:, 1]
print(f"{'class_weight':22s} PR-AUC {average_precision_score(yte, p):.3f}  @0.5: recall {recall_score(yte, p>.5):.2f} prec {precision_score(yte, p>.5):.2f}")
# the leakage mistake: SMOTE before the split -> CV score inflated
from sklearn.model_selection import cross_val_score, StratifiedKFold
Xp = X[y == 1]; new = smote(Xp, (y == 0).sum() - len(Xp)); Xa = np.r_[X, new]; ya = np.r_[y, np.ones(len(new), int)]
s = cross_val_score(RandomForestClassifier(100, random_state=0, n_jobs=-1), Xa, ya, cv=StratifiedKFold(5, shuffle=True, random_state=0), scoring="average_precision")
print("WRONG (SMOTE before CV) PR-AUC", s.mean().round(3))

train positives 102 of 3000
baseline               PR-AUC 0.846  @0.5: recall 0.68 prec 0.92
SMOTE to 50/50         PR-AUC 0.783  @0.5: recall 0.82 prec 0.67
class_weight           PR-AUC 0.811  @0.5: recall 0.50 prec 0.85
WRONG (SMOTE before CV) PR-AUC 0.998


## Chapter recap

## Practice set

Problem 12.P.1 (easy) 10,000 transactions, 50 fraud. A model flags 40 payments, of which 30 are fraud. Compute precision and recall.

## Prompt pack

---
Done? Do the **Practice set** at the end of the chapter, then clear your Review cards on the site.

https://dadipavan.github.io/Learning/study/ml/ch12.html